# Example 11: World Port Index

Neptune ships a built-in dictionary of 3,800+ ports from the NGA World
Port Index, plus 11,700 UNLOCODE entries and 285 EEZ regions. This
example walks through the complete port intelligence workflow:

1. Search and explore the port dictionary
2. Detect port calls with zero configuration
3. Enrich detected events with port metadata
4. Derive port polygons from AIS data
5. Visualize port boundaries on a map
6. Resolve vessel destination fields

## Prerequisites

```bash
pip install neptune-ais[geo]
```

The port dictionary ships in-package (no download needed).
Port-call detection and polygon derivation require position data
from [Example 02](02_archival_ingest.ipynb).

## 1. Explore the Port Dictionary

In [ ]:
from neptune_ais.ports import index

pi = index()
print(f"Ports: {len(pi.ports):,}")
print(f"UNLOCODEs: {len(pi.unlocodes):,}")
print(f"EEZ regions: {len(pi.eez)}")

### Full-text search

In [ ]:
pi.search("Rotterdam")

### Nearby ports

In [ ]:
# Ports within 50 km of central Rotterdam
pi.near(51.9, 4.5, radius_km=50)

### Lookup by UNLOCODE or WPI number

In [ ]:
port = pi.by_unlocode("NLRTM")
print(f"{port.name} ({port.unlocode})")
print(f"  Harbor size: {port.harbor_size}")
print(f"  Coordinates: {port.lat:.4f}, {port.lon:.4f}")
print(f"  Pilotage: {port.has_pilotage}, Cranes: {port.has_cranes}")

### Filter by country

In [ ]:
# All Dutch ports
pi.by_country("NL")

### Filter by facilities

In [ ]:
import polars as pl

# Large ports with drydock facilities
pi.with_facilities(has_drydock=True).filter(
    pl.col("harbor_size") == "L"
).select("name", "unlocode", "country_code", "harbor_size")

## 2. Zero-Config Port Call Detection

The `port_calls()` helper auto-loads WPI boundaries and uses
vectorized spatial matching. No manual boundary loading needed.

In [ ]:
from neptune_ais.api import Neptune

# Requires downloaded data — see Example 02
# n = Neptune("2024-06-15", sources=["noaa"])
# events = n.port_calls()
# print(f"Detected {len(events)} port call(s)")
# events.head()

## 3. Enrich Port Calls with Metadata

Detected port-call events can be enriched with WPI metadata:
UNLOCODE, country, harbor size, facilities, depths.

In [ ]:
from neptune_ais.helpers import enrich_port_calls

# enrich=True is the default in port_calls(), but you can
# also enrich separately:
#
# raw_events = n.port_calls(enrich=False)
# enriched = enrich_port_calls(raw_events)
# enriched.select(
#     "mmsi", "start_time", "end_time",
#     "port_name", "unlocode", "country_code",
#     "harbor_size", "has_cranes", "has_fuel",
# )

## 4. Derive Port Polygons (Tier 2)

The Tier 2 pipeline derives empirical port boundary polygons
from AIS position data. It filters low-speed positions, assigns
them to nearby WPI ports, computes concave hulls, splits into
spatial zones, and scores confidence.

Requires `shapely >= 2.0` (the `[geo]` extra).

In [ ]:
# Requires downloaded position data
# polygons = n.derive_port_polygons()
# print(f"Derived {len(polygons)} zone(s) across"
#       f" {polygons['port_name'].n_unique()} port(s)")
# polygons.select(
#     "port_name", "zone_id", "confidence",
#     "position_count", "vessel_count",
# ).head(10)

## 5. Visualize Port Boundaries

`prepare_ports()` returns port centers and polygon boundaries
ready for map rendering. Tier 2 derived polygons are preferred;
remaining ports fall back to Tier 1 bbox rectangles.

In [ ]:
from neptune_ais.viz import prepare_ports, Viewport

# Rotterdam area viewport
vp = Viewport(west=3.5, south=51.5, east=5.0, north=52.2)
layers = prepare_ports(viewport=vp)

print(f"Centers: {len(layers['centers'])} ports")
print(f"Polygons: {len(layers['polygons'])} boundaries")
layers["centers"]

In [ ]:
layers["polygons"].select(
    "name", "polygon_source", "confidence",
    "bbox_west", "bbox_south", "bbox_east", "bbox_north",
)

## 6. Resolve Vessel Destinations

The AIS destination field (Message Type 5) is free-text and noisy.
The destination resolver fuzzy-matches it to canonical ports.

In [ ]:
from neptune_ais.ports._destination import resolve_destination

# Exact UNLOCODE
port = resolve_destination("NLRTM", pi)
print(f"NLRTM -> {port.name}")

# Fuzzy match
port = resolve_destination("ROTT", pi)
print(f"ROTT -> {port.name if port else 'no match'}")

# Name with noise
port = resolve_destination("PORT OF ROTTERDAM", pi)
print(f"PORT OF ROTTERDAM -> {port.name if port else 'no match'}")

In [ ]:
import polars as pl
from neptune_ais.ports._destination import resolve_destination_column

# Batch resolution
destinations = pl.Series([
    "NLRTM", "ROTTERDAM", "HAMBURG", "PORT OF ANTWERP",
    "ROTT", None, "", ">>GARBAGE<<",
])
resolved = resolve_destination_column(destinations, pi)
pl.concat([destinations.alias("raw_destination"), resolved], how="horizontal")

## CLI Access

All port queries are also available from the command line:

```bash
neptune ports search "Rotterdam"
neptune ports near 51.9 4.5
neptune ports info NLRTM
neptune ports country NL
neptune ports derive -d 2024-06-15
neptune ports export -o ports.geojson
```

## Next Steps

- **[08 — Spatial Visualization](08_spatial_visualization.ipynb)**: Interactive maps with lonboard
- **[HEURISTICS.md](../HEURISTICS.md)**: Detection assumptions and known limitations
- **Port polygon derivation** improves with more data — run `neptune ports derive` periodically